In [ ]:
import os
import numpy as np
import pandas as pd
from to_numpy import *

def get_dataarray(config):
    match config['dataset_name']:
        case 'metr_la' | 'pems_bay':
            data_array, date_array = metrla_pemsbay(config['raw_data_path'])
        case 'pems04' | 'pems08':
            data_array, date_array = npz_file_pems0408(config['raw_data_path'], 
                                                       start_time=config['start_time'], 
                                                       time_windows=config['time_window'], 
                                                       lens=config['lens'])
        case 'taxibj13' | 'taxibj14' | 'taxibj15' | 'taxibj16':
            data_array, date_array = taxibj(config['raw_data_path'])
        case 'ettm1' | 'ettm2':
            data_array, date_array = ett(config['raw_data_path'])
        case 'exchange_rate':
            data_array, date_array = exchange_rate(config['raw_data_path'])
        case 'illness':
            data_array, date_array = illness(config['raw_data_path'])
        case 'traffic':
            data_array, date_array = traffic(config['raw_data_path'])
        case 'weather':
            data_array, date_array = weather(config['raw_data_path'])
        case 'electricity':
            data_array, date_array = electricity(config['raw_data_path'])
        case _:
            raise KeyError(f'dataset "{config["dataset_name"]}" not defined.')

    return data_array, date_array

In [ ]:
def save_dataarray(config, data_array, date_array):
    # data 数据格式统一
    if isinstance(data_array, (pd.DataFrame, pd.Series)):
        data_array = data_array.values
    else:
        data_array = np.asarray(data_array)
    # date 数据格式统一
    if isinstance(date_array, (pd.DatetimeIndex, pd.Series)):
        date_array = date_array.values
    else:
        date_array = np.asarray(date_array)

    parsed_dates = pd.to_datetime(date_array, format='mixed', errors='coerce')
    date_array = parsed_dates.values.astype('datetime64[s]')

    # 验证数据形状
    assert data_array.shape[0] == date_array.shape[0], "数据和日期长度不匹配"
    assert data_array.shape[0] == config['lens'], f"数据长度 {data_array.shape[0]} 与配置长度 {config['lens']} 不匹配"
    assert data_array.shape[1] == config['num_nodes'], f"数据节点数 {data_array.shape[1]} 与配置节点数 {config['num_nodes']} 不匹配"
    assert data_array.shape[2] == config['num_channels'], f"数据通道数 {data_array.shape[2]} 与配置通道数 {config['num_channels']} 不匹配"
    
    os.makedirs(config['bin_data_dir'], exist_ok=True)
    np.save(config['bin_data_path'], data_array)
    np.save(config['date_data_path'], date_array)

    new_data_array = np.load(config['bin_data_path'], allow_pickle=True)
    new_date_array = np.load(config['date_data_path'], allow_pickle=True)

    # 验证保存数据一致性
    assert np.array_equal(data_array, new_data_array)
    assert np.array_equal(date_array, new_date_array)
    return new_data_array, new_date_array

In [ ]:
# metr_la dataset
config = {
    'dataset_name': 'metr_la',
    'raw_data_path': './storage/datasets/metr_la/metr_la.h5',
    'bin_data_dir': './storage/datasets/metr_la/bin_data',
    'bin_data_path': './storage/datasets/metr_la/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/metr_la/bin_data/date_array.npy',
    'start_time': '2012-03-01 00:00:00',
    'end_time': '2012-06-27 23:55:00',
    'time_window': 5,
    'lens': 34272,
    'num_nodes': 207,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# pems_bay dataset
config = {
    'dataset_name': 'pems_bay',
    'raw_data_path': './storage/datasets/pems_bay/pems_bay.h5',
    'bin_data_dir': './storage/datasets/pems_bay/bin_data',
    'bin_data_path': './storage/datasets/pems_bay/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/pems_bay/bin_data/date_array.npy',
    'start_time': '2017-01-01 00:00:00',
    'end_time': '2017-06-30 23:55:00',
    'time_window': 5,
    'lens': 52116,
    'num_nodes': 325,
    'num_channels': 1,
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# pems04 dataset
config = {
    'dataset_name': 'pems04',
    'raw_data_path': './storage/datasets/pems04/pems04.npz',
    'bin_data_dir': './storage/datasets/pems04/bin_data',
    'bin_data_path': './storage/datasets/pems04/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/pems04/bin_data/date_array.npy',
    'start_time': '2017-01-01 00:00:00',
    'end_time': '2017-02-28 23:55:00',
    'time_window': 5,
    'lens': 16992,
    'num_nodes': 307,
    'num_channels': 3
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# pems08 dataset
config = {
    'dataset_name': 'pems08',
    'raw_data_path': './storage/datasets/pems08/pems08.npz',
    'bin_data_dir': './storage/datasets/pems08/bin_data',
    'bin_data_path': './storage/datasets/pems08/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/pems08/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2016-08-31 23:55:00',
    'time_window': 5,
    'lens': 17856,
    'num_nodes': 170,
    'num_channels': 3

}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# taxibj13 dataset
config = {
    'dataset_name': 'taxibj13',
    'raw_data_path': './storage/datasets/TaxiBJ/BJ13_M32x32_T30_InOut.h5',
    'bin_data_dir': './storage/datasets/TaxiBJ/taxibj13/bin_data',
    'bin_data_path': './storage/datasets/TaxiBJ/taxibj13/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/TaxiBJ/taxibj13/bin_data/date_array.npy',
    'start_time': '2013-07-01 00:00:00',
    'end_time': '2013-10-29 23:30:00',
    'time_window': 30,
    'lens': 4888,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# taxibj14 dataset
config = {
    'dataset_name': 'taxibj14',
    'raw_data_path': './storage/datasets/TaxiBJ/BJ14_M32x32_T30_InOut.h5',
    'bin_data_dir': './storage/datasets/TaxiBJ/taxibj14/bin_data',
    'bin_data_path': './storage/datasets/TaxiBJ/taxibj14/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/TaxiBJ/taxibj14/bin_data/date_array.npy',
    'start_time': '2014-03-01 00:00:00',
    'end_time': '2014-06-27 11:30:00',
    'time_window': 30,
    'lens': 4780,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# taxibj15 dataset
config = {
    'dataset_name': 'taxibj15',
    'raw_data_path': './storage/datasets/TaxiBJ/BJ15_M32x32_T30_InOut.h5',
    'bin_data_dir': './storage/datasets/TaxiBJ/taxibj15/bin_data',
    'bin_data_path': './storage/datasets/TaxiBJ/taxibj15/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/TaxiBJ/taxibj15/bin_data/date_array.npy',
    'start_time': '2015-03-01 00:00:00',
    'end_time': '2015-06-30 23:30:00',
    'time_window': 30,
    'lens': 5596,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# taxibj16 dataset
config = {
    'dataset_name': 'taxibj16',
    'raw_data_path': './storage/datasets/TaxiBJ/BJ16_M32x32_T30_InOut.h5',
    'bin_data_dir': './storage/datasets/TaxiBJ/taxibj16/bin_data',
    'bin_data_path': './storage/datasets/TaxiBJ/taxibj16/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/TaxiBJ/taxibj16/bin_data/date_array.npy',

    'time_window': 30,
    'lens': 7220,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# ettm1 dataset
config = {
    'dataset_name': 'ettm1',
    'raw_data_path': './storage/datasets/ETT-small/ETTm1.csv',
    'bin_data_dir': './storage/datasets/ETT-small/ettm1/bin_data',
    'bin_data_path': './storage/datasets/ETT-small/ettm1/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/ETT-small/ettm1/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2018-06-26 19:45:00',
    'time_window': 15,
    'lens': 69680,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# ettm2 dataset
config = {
    'dataset_name': 'ettm2',
    'raw_data_path': './storage/datasets/ETT-small/ETTm2.csv',
    'bin_data_dir': './storage/datasets/ETT-small/ettm2/bin_data',
    'bin_data_path': './storage/datasets/ETT-small/ettm2/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/ETT-small/ettm2/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2018-06-26 19:45:00',
    'time_window': 15,
    'lens': 69680,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# exchange_rate dataset
config = {
    'dataset_name': 'exchange_rate',
    'raw_data_path': './storage/datasets/exchange_rate/exchange_rate.csv',
    'bin_data_dir': './storage/datasets/exchange_rate/bin_data',
    'bin_data_path': './storage/datasets/exchange_rate/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/exchange_rate/bin_data/date_array.npy',
    'start_time': '1990-01-01 00:00:00',
    'end_time': '2010-10-10 00:00:00',
    'time_window': 1440,
    'lens': 7588,
    'num_nodes': 1,
    'num_channels': 8
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# illness dataset
config = {
    'dataset_name': 'illness',
    'raw_data_path': './storage/datasets/illness/national_illness.csv',
    'bin_data_dir': './storage/datasets/illness/bin_data',
    'bin_data_path': './storage/datasets/illness/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/illness/bin_data/date_array.npy',
    'start_time': '2002-01-01 00:00:00',
    'end_time': '2020-06-30 00:00:00',
    'time_window': 10080,
    'lens': 966,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# traffic dataset
config = {
    'dataset_name': 'traffic',
    'raw_data_path': './storage/datasets/traffic/traffic.csv',
    'bin_data_dir': './storage/datasets/traffic/bin_data',
    'bin_data_path': './storage/datasets/traffic/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/traffic/bin_data/date_array.npy',
    'start_time': '2016-07-01 02:00:00',
    'end_time': '2018-07-02 01:00:00',
    'time_window': 60,
    'lens': 17544,
    'num_nodes': 862,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# weather dataset
config = {
    'dataset_name': 'weather',
    'raw_data_path': './storage/datasets/weather/weather.csv',
    'bin_data_dir': './storage/datasets/weather/bin_data',
    'bin_data_path': './storage/datasets/weather/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/weather/bin_data/date_array.npy',
    'start_time': '2020-01-01 00:10:00',
    'end_time': '2021-01-01 00:00:00',
    'time_window': 10,
    'lens': 52696,
    'num_nodes': 1,
    'num_channels': 21
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")

In [ ]:
# electricity dataset
config = {
    'dataset_name': 'electricity',
    'raw_data_path': './storage/datasets/electricity/electricity.csv',
    'bin_data_dir': './storage/datasets/electricity/bin_data',
    'bin_data_path': './storage/datasets/electricity/bin_data/data_array.npy',
    'date_data_path': './storage/datasets/electricity/bin_data/date_array.npy',
    'start_time': '2016-07-01 02:00:00',
    'end_time': '2019-07-02 01:00:00',
    'time_window': 60,
    'lens': 26304,
    'num_nodes': 321,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"数据数组形状: {new_data_array.shape}")
print(f"起始时间: {new_date_array[0]}, 结束时间: {new_date_array[-1]}")